# 토크나이저 분석 — MAPF-GPT

논문의 핵심 기여 중 하나인 **67개 토큰 어휘 기반 토크나이제이션**이 코드에서 어떻게 구현되어 있는지 분석합니다.

In [ ]:
import sys
sys.path.insert(0, '/content/MAPF-GPT')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. 어휘(Vocabulary) 구성 분석

논문 설명: *"총 67개 토큰으로 구성된 어휘"*

In [ ]:
# 67개 토큰의 구성
vocab_structure = {
    '수치값 토큰 [-20, 20]': 41,   # 정수 수치 범위
    '범위 초과 토큰':         2,   # >20, <-20
    '무한대 토큰 (장애물)':   1,   # ∞ (blocked cell)
    '행동 토큰':              5,   # 상/하/좌/우/대기
    'Greedy action 토큰':    16,  # 복합 방향 포함 (up-right 등)
    'Empty 토큰':             1,   # 패딩 및 빈 슬롯
    'Empty action 토큰':      1,   # 에피소드 시작 시 행동 없음
}

total = sum(vocab_structure.values())
print(f"{'토큰 종류':<30} {'개수':>5}")
print("-" * 38)
for k, v in vocab_structure.items():
    print(f"{k:<30} {v:>5}")
print("-" * 38)
print(f"{'합계':<30} {total:>5}")

# 시각화
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2196F3','#03A9F4','#00BCD4','#4CAF50','#FF9800','#9C27B0','#F44336']
wedges, texts, autotexts = ax.pie(
    vocab_structure.values(),
    labels=[f"{k}\n({v}개)" for k, v in vocab_structure.items()],
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
    textprops={'fontsize': 9}
)
ax.set_title('MAPF-GPT 토큰 어휘 구성 (총 67개)', fontsize=13)
plt.tight_layout()
plt.savefig('vocab_structure.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. 입력 시퀀스 구조 (256 토큰)

In [ ]:
# 256 토큰의 구성 시각화
fig, ax = plt.subplots(figsize=(14, 3))

# 구간 정의
segments = [
    (0,   121, '#2196F3', 'Cost-to-go\n(11×11 시야)\n121 토큰'),
    (121, 131, '#4CAF50', '에이전트 0\n(자신)\n10토큰'),
    (131, 141, '#66BB6A', '에이전트 1\n10토큰'),
    (141, 151, '#81C784', '에이전트 2\n10토큰'),
    (151, 161, '#A5D6A7', '...'),
    (161, 251, '#C8E6C9', '에이전트 3~12\n(각 10토큰, 최대 13명)'),
    (251, 256, '#FF9800', 'Padding\n5토큰'),
]

for start, end, color, label in segments:
    ax.barh(0, end - start, left=start, height=0.5, color=color, edgecolor='white', linewidth=1.5)
    if end - start > 8:
        ax.text((start + end) / 2, 0, label,
                ha='center', va='center', fontsize=8, fontweight='bold')

ax.set_xlim(0, 256)
ax.set_ylim(-0.5, 0.8)
ax.set_xlabel('토큰 인덱스', fontsize=11)
ax.set_title('MAPF-GPT 입력 시퀀스 구조 (256 토큰)', fontsize=12)
ax.set_yticks([])
ax.grid(axis='x', alpha=0.3)

# 구간 경계 표시
for pos in [0, 121, 251, 256]:
    ax.axvline(x=pos, color='gray', linestyle='--', alpha=0.5)
    ax.text(pos, 0.35, str(pos), ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('token_sequence_structure.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 에이전트 1명당 토큰 구성 (10 토큰)

In [ ]:
agent_tokens = {
    '현재 위치 x': 1,
    '현재 위치 y': 1,
    '목표 위치 x': 1,
    '목표 위치 y': 1,
    '행동 이력 t-1': 1,
    '행동 이력 t-2': 1,
    '행동 이력 t-3': 1,
    '행동 이력 t-4': 1,
    '행동 이력 t-5': 1,
    'Greedy action': 1,
}

print("에이전트 1명당 토큰 구성 (총 10토큰):")
print("-" * 35)
for i, (k, v) in enumerate(agent_tokens.items()):
    print(f"  [{i:2d}] {k}")

print(f"\n최대 수용 에이전트: 135 ÷ 10 = 13명 (egocentric 포함)")
print(f"나머지 5토큰은 padding으로 처리")

## 4. Cost-to-go 값의 의미

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# 간단한 Cost-to-go 맵 예시 (11×11)
# 중심(5,5)에 에이전트, (0,0)에 목표 가정
grid_size = 11
center = (5, 5)
goal = (0, 0)

cost_to_go = np.zeros((grid_size, grid_size))
for i in range(grid_size):
    for j in range(grid_size):
        # 절대 cost-to-go (맨해튼 거리 근사)
        abs_cost = abs(i - goal[0]) + abs(j - goal[1])
        center_cost = abs(center[0] - goal[0]) + abs(center[1] - goal[1])
        # 상대적 cost-to-go (에이전트 위치 기준 정규화)
        cost_to_go[i][j] = abs_cost - center_cost

# 몇 개 셀 장애물로 설정
obstacles = [(2,2),(2,3),(2,4),(3,4),(4,4),(4,5),(4,6)]
for r, c in obstacles:
    cost_to_go[r][c] = np.inf

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 왼쪽: cost-to-go 히트맵
display = np.where(np.isinf(cost_to_go), np.nan, cost_to_go)
cmap = plt.cm.RdYlGn_r
cmap.set_bad('black')

im = axes[0].imshow(display, cmap=cmap, vmin=-10, vmax=10)
axes[0].scatter(*center[::-1], s=200, c='blue', marker='o', zorder=5, label='에이전트')
axes[0].scatter(*goal[::-1], s=200, c='gold', marker='*', zorder=5, label='목표')
axes[0].set_title('Cost-to-go 맵 (에이전트 위치 기준 상대값)', fontsize=11)
axes[0].legend(fontsize=10)
plt.colorbar(im, ax=axes[0], label='Cost-to-go 상대값')

# 오른쪽: 토큰화 결과
clipped = np.clip(display, -20, 20)
clipped = np.where(np.isnan(display), 999, clipped)  # ∞ → 999로 표시

axes[1].imshow(np.where(clipped == 999, np.nan, clipped), cmap=cmap, vmin=-20, vmax=20)
for i in range(grid_size):
    for j in range(grid_size):
        val = clipped[i][j]
        text = '∞' if val == 999 else f'{int(val):+d}'
        color = 'white' if val == 999 else 'black'
        axes[1].text(j, i, text, ha='center', va='center', fontsize=7, color=color)
axes[1].set_title('토큰화된 값 ([-20,20] 클리핑, 장애물=∞)', fontsize=11)

plt.tight_layout()
plt.savefig('cost_to_go_visualization.png', dpi=150, bbox_inches='tight')
plt.show()